In [4]:
import os
 
from dotenv import load_dotenv
from openai import OpenAI
 
load_dotenv()
 
api_key = os.getenv("DEEPSEEK_API_KEY")
 
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)
 
print(api_key is not None)

True


In [7]:
import sys
import os
from dotenv import load_dotenv
from importlib.metadata import version

load_dotenv()

print(f"Python: {sys.version}")

try:
    import sklearn
    print(f"scikit-learn: {sklearn.__version__} ✓")
except ImportError:
    print("scikit-learn no instalado — ejecuta: pip install scikit-learn")

try:
    import openai
    print(f"openai: {version('openai')} ✓")
except ImportError:
    print("openai no instalado — ejecuta: pip install openai")

api_key = os.getenv("DEEPSEEK_API_KEY")

if api_key:
    print(f"DEEPSEEK_API_KEY: configurada ✓ (termina en ...{api_key[-4:]})")
else:
    print("DEEPSEEK_API_KEY: NO configurada ⚠ — revisa el archivo .env")

Python: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
scikit-learn: 1.9.0 ✓
openai: 2.46.0 ✓
DEEPSEEK_API_KEY: configurada ✓ (termina en ...8626)


In [14]:
from collections import Counter

MENSAJES = [

    # --- horario (10 mensajes) ---
    ("¿A qué hora abren los sábados?", "horario"),
    ("¿Cuál es el horario de atención al cliente?", "horario"),
    ("¿Hasta qué hora atienden los domingos?", "horario"),
    ("¿Trabajan el 20 de julio?", "horario"),
    ("¿Las oficinas abren el lunes festivo?", "horario"),
    ("Necesito saber cuándo abren el banco", "horario"),
    ("¿Hay atención los días festivos?", "horario"),
    ("¿A qué hora cierran la sucursal de El Poblado?", "horario"),
    ("¿Tienen atención los fines de semana?", "horario"),
    ("¿Qué días están disponibles los asesores?", "horario"),

    # --- queja (10 mensajes) ---
    ("Me debitaron dos veces el mismo pago", "queja"),
    ("No reconozco un cobro en mi extracto", "queja"),
    ("Llevo 3 días sin poder usar mi tarjeta débito", "queja"),
    ("El cajero se tragó mi tarjeta y no me la devolvió", "queja"),
    ("Me bloquearon la cuenta sin avisarme", "queja"),
    ("Hice una transferencia y el dinero no llegó", "queja"),
    ("Me cobraron una comisión que no autoricé", "queja"),
    ("Llevo una hora en la línea y nadie me atiende", "queja"),
    ("Mi solicitud de crédito lleva 15 días sin respuesta", "queja"),
    ("El app del banco no me deja ingresar desde ayer", "queja"),

    # --- producto (10 mensajes) ---
    ("¿Cuál es la tasa de interés del crédito de consumo?", "producto"),
    ("¿Qué documentos necesito para abrir una cuenta de ahorros?", "producto"),
    ("¿Cuánto es el cupo mínimo de la tarjeta de crédito?", "producto"),
    ("¿Tienen CDTs a 90 días?", "producto"),
    ("¿Qué beneficios tiene la tarjeta Visa Platinum?", "producto"),
    ("¿Puedo pedir un préstamo si soy independiente?", "producto"),
    ("¿Cuánto tiempo tardan en desembolsar un crédito?", "producto"),
    ("¿Qué seguro incluye la cuenta corriente?", "producto"),
    ("¿Tienen cuenta de nómina para empresas pequeñas?", "producto"),
    ("¿Cuál es el monto máximo del giro internacional?", "producto"),

    # --- saludo (10 mensajes) ---
    ("Hola buenos días, necesito ayuda", "saludo"),
    ("Buenas tardes, ¿me pueden atender?", "saludo"),
    ("Hola, ¿están disponibles?", "saludo"),
    ("Buen día", "saludo"),
    ("Buenos días, tengo una pregunta", "saludo"),
    ("Hola, cómo están", "saludo"),
    ("Buenas noches, requiero asistencia", "saludo"),
    ("Hey, ¿alguien me puede ayudar?", "saludo"),
    ("Hola, estoy aquí", "saludo"),
    ("Bienvenidos, soy cliente nuevo", "saludo"),
]

print(f"Total de mensajes: {len(MENSAJES)}")

conteo = Counter(etiqueta for _, etiqueta in MENSAJES)

for categoria, cantidad in sorted(conteo.items()):
    print(f"{categoria}: {cantidad}")

Total de mensajes: 40
horario: 10
producto: 10
queja: 10
saludo: 10


In [15]:
# Celda 3 — Clasificador simbólico

def clasificar_simbolico(mensaje: str) -> str:
    """
    Clasifica la intención del mensaje usando reglas IF-THEN.
    No usa ningún dato de entrenamiento.
    """

    texto = mensaje.lower()

    # ── REGLAS PARA 'horario' ────────────────────────────────
    palabras_horario = [
        "hora", "horario", "abren", "cierran", "atienden", "atención",
        "disponibles", "festivo", "sábado", "domingo", "lunes", "días"
    ]
    if any(p in texto for p in palabras_horario):
        return "horario"

    # ── REGLAS PARA 'queja' ──────────────────────────────────
    palabras_queja = [
        "no reconozco", "cobro", "debit", "bloquearon", "bloqueada",
        "no llegó", "no me deja", "se tragó", "sin respuesta",
        "no pude", "no puedo", "error", "falla", "problema",
        "dos veces", "no funciona", "días sin", "sin avisarme",
        "transferencia", "comisión", "queja", "reclamo"
    ]
    if any(p in texto for p in palabras_queja):
        return "queja"

    # ── REGLAS PARA 'producto' ───────────────────────────────
    palabras_producto = [
        "tasa", "interés", "crédito", "cuenta", "tarjeta", "cdt",
        "préstamo", "documentos", "cupo", "beneficios", "seguro",
        "nómina", "giro", "desembolsar", "monto", "visa", "platinum",
        "independiente", "ahorros", "corriente"
    ]
    if any(p in texto for p in palabras_producto):
        return "producto"

    # ── REGLAS PARA 'saludo' ─────────────────────────────────
    palabras_saludo = [
        "hola", "buenos", "buenas", "buen día", "hey", "bienvenidos"
    ]
    if any(p in texto for p in palabras_saludo):
        return "saludo"

    return "desconocido"

In [16]:
# Celda 4 — Evaluación simbólico

import time
from collections import Counter

def evaluar_clasificador(fn_clasificar, mensajes, nombre):
    """Evalúa cualquier función clasificadora y devuelve métricas."""
    correctos = 0
    errores = []
    tiempos = []

    for texto, esperado in mensajes:
        t0 = time.perf_counter()
        pred = fn_clasificar(texto)
        t1 = time.perf_counter()

        tiempos.append(t1 - t0)
        if pred == esperado:
            correctos += 1
        else:
            errores.append((texto, esperado, pred))

    precision = correctos / len(mensajes)
    latencia_ms = sum(tiempos) / len(tiempos) * 1000

    print(f"\n{'-' * 55}")
    print(f"  {nombre}")
    print(f"{'-' * 55}")
    print(f"  Precisión:   {correctos}/{len(mensajes)} = {precision*100:.1f}%")
    print(f"  Latencia:    {latencia_ms:.3f} ms (promedio por mensaje)")
    print(f"\n  Errores ({len(errores)}):")
    for texto, esperado, pred in errores:
        print(f"    X [{esperado}]->[{pred}] | '{texto[:50]}'")

    return {
        "precision": precision,
        "latencia_ms": latencia_ms,
        "errores": errores
    }


resultados_simbolico = evaluar_clasificador(
    clasificar_simbolico,
    MENSAJES,
    "SIMBÓLICO — Reglas IF-THEN"
)


-------------------------------------------------------
  SIMBÓLICO — Reglas IF-THEN
-------------------------------------------------------
  Precisión:   32/40 = 80.0%
  Latencia:    0.003 ms (promedio por mensaje)

  Errores (8):
    X [horario]->[desconocido] | '¿Trabajan el 20 de julio?'
    X [queja]->[horario] | 'Llevo 3 días sin poder usar mi tarjeta débito'
    X [queja]->[horario] | 'Llevo una hora en la línea y nadie me atiende'
    X [queja]->[horario] | 'Mi solicitud de crédito lleva 15 días sin respuest'
    X [producto]->[horario] | '¿Tienen CDTs a 90 días?'
    X [saludo]->[horario] | 'Hola buenos días, necesito ayuda'
    X [saludo]->[horario] | 'Hola, ¿están disponibles?'
    X [saludo]->[horario] | 'Buenos días, tengo una pregunta'


## ¿Cuantos errores cometio el sistema simbolico? ------- 8 Errores.
## ¿Que tiene en comun los mensajes de fallo? ------- En su mayoria eran mensajes sobre el horario.
## ¿Cuantas reglas adicionales necesitas para cumplir esos casos? ------- Solo se deberia agregar 1 regla adicional para que todas tengan una clasificacion.

In [17]:
# Celda 5 — Preparación del dataset para sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Separar textos y etiquetas
textos = [m[0] for m in MENSAJES]
etiquetas = [m[1] for m in MENSAJES]

# División 75% entrenamiento / 25% prueba — estratificada (mismas proporciones por clase)
X_train, X_test, y_train, y_test = train_test_split(
    textos, etiquetas,
    test_size=0.25,
    random_state=42,
    stratify=etiquetas
)

print(f"Entrenamiento: {len(X_train)} mensajes")
print(f"Prueba:        {len(X_test)} mensajes")

from collections import Counter
print(f"Clases en prueba: {Counter(y_test)}")

Entrenamiento: 30 mensajes
Prueba:        10 mensajes
Clases en prueba: Counter({'horario': 3, 'queja': 3, 'producto': 2, 'saludo': 2})


In [18]:
# Celda 6 — Entrenamiento

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # uni-gramas y bi-gramas
    min_df=1,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=1.0
)

clf.fit(X_train_vec, y_train)

print("Modelo entrenado ✓")
print(f"Vocabulario: {len(vectorizer.vocabulary_)} términos")

Modelo entrenado ✓
Vocabulario: 274 términos


In [19]:
# Celda 7 — Evaluación estadístico

def clasificar_estadistico(mensaje: str) -> str:
    vec = vectorizer.transform([mensaje])
    return clf.predict(vec)[0]

resultados_estadistico = evaluar_clasificador(
    clasificar_estadistico,
    MENSAJES,
    "ESTADÍSTICO — TF-IDF + Logistic Regression"
)

# Reporte detallado por clase
y_pred_test = clf.predict(X_test_vec)

print(f"\n{'-' * 55}")
print("  Reporte en conjunto de prueba (10 mensajes):")
print(f"{'-' * 55}")

print(classification_report(y_test, y_pred_test))


-------------------------------------------------------
  ESTADÍSTICO — TF-IDF + Logistic Regression
-------------------------------------------------------
  Precisión:   39/40 = 97.5%
  Latencia:    0.515 ms (promedio por mensaje)

  Errores (1):
    X [saludo]->[producto] | 'Bienvenidos, soy cliente nuevo'

-------------------------------------------------------
  Reporte en conjunto de prueba (10 mensajes):
-------------------------------------------------------
              precision    recall  f1-score   support

     horario       1.00      1.00      1.00         3
    producto       0.67      1.00      0.80         2
       queja       1.00      1.00      1.00         3
      saludo       1.00      0.50      0.67         2

    accuracy                           0.90        10
   macro avg       0.92      0.88      0.87        10
weighted avg       0.93      0.90      0.89        10



## ¿El modelo estadístico cometió más o menos errores que el simbólico? -------
## ¿Los errores son los mismos o diferentes? -------
## El modelo se entrenó con 30 ejemplos (7-8 por clase). ¿Es suficiente? ¿Qué pasaría con 300? -------

## Pruebas

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError("No se encontró DEEPSEEK_API_KEY en el archivo .env")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    temperature=0.7,
    max_tokens=512,
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Explícame los números primos."}
    ]
)

print(response.choices[0].message.content)

¡Claro! Los números primos son uno de los conceptos más fascinantes y fundamentales en matemáticas. Aquí te lo explico de forma sencilla:

**Definición:**  
Un número primo es un número natural mayor que 1 que solo tiene dos divisores exactos: el 1 y él mismo.  
Es decir, no se puede dividir exactamente (con resto cero) por ningún otro número aparte de 1 y el propio número.

**Ejemplos clave:**  
- **Sí son primos:** 2, 3, 5, 7, 11, 13, 17, 19, 23...  
  - El 2 es el único número primo par (si fuera par y mayor que 2, sería divisible entre 2).  
- **No son primos (compuestos):** 4 (divisible entre 2), 6 (entre 2 y 3), 9 (entre 3), 15 (entre 3 y 5).  
- **Casos especiales:**  
  - El 1 **no** se considera primo porque solo tiene un divisor (el 1 mismo).  
  - Los números mayores que 1 que no son primos se llaman **compuestos**.

**¿Cómo saber si un número es primo?**  
Para números pequeños, puedes probar dividirlo entre números primos menores que su raíz cuadrada. Por ejemplo:  
- ¿Es 

In [12]:
from pprint import pprint

pprint(response.model_dump())

NameError: name 'response' is not defined